In [59]:
from pathlib import Path
import pandas as pd

parent_dir = Path.cwd().parent
file_dir = parent_dir / "Extract"

file_list = list(file_dir.glob("POSAS*"))

FASCE = [
    ("0-17", 0, 17),
    ("18-44", 18, 44),
    ("45-69", 45, 69),
    ("70+",   70, None),
]

SEP = ";"

COL_PROV = "Provincia"

COL_CODICE = "Codice comune"
COL_COMUNE = "Comune"
COL_ETA    = "Età"
COL_M      = "Totale maschi"
COL_F      = "Totale femmine"
COL_TOT    = "Totale"

ETA_TOTALE_COMUNE = 999

In [60]:
def leggi_file_posas(path: str) -> pd.DataFrame:
    
    df = pd.read_csv(
        path,
        sep=SEP,
        encoding="utf-8-sig",
        dtype=str
    )

    for col in (COL_ETA, COL_M, COL_F, COL_TOT):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[df[COL_ETA] != ETA_TOTALE_COMUNE].copy()

    for col in (COL_M, COL_F, COL_TOT):
        df[col] = df[col].fillna(0).astype(int)

    df[COL_PROV] = path.name.split("_",4)[4][:-4].replace("_", " ")

    return df


def assegna_fascia(eta: int) -> str:
    for etichetta, emin, emax in FASCE:
        if eta < emin:
            continue
        if emax is None or eta <= emax:
            return etichetta

In [61]:
frames = []

for path in file_list:
    df = leggi_file_posas(path)
    frames.append(df)

dati = pd.concat(frames, ignore_index=True)

dati["Fascia"] = dati[COL_ETA].apply(assegna_fascia)

out = (dati.groupby([COL_CODICE, COL_PROV, COL_COMUNE, "Fascia"])
            .agg(Maschi=(COL_M, "sum"),
                    Femmine=(COL_F, "sum"),
                    Totale=(COL_TOT, "sum"))
            .reset_index())

ordine_fasce = [f[0] for f in FASCE]
out["Fascia"] = pd.Categorical(out["Fascia"], categories=ordine_fasce, ordered=True)
out = out.sort_values([COL_CODICE, "Fascia"]).reset_index(drop=True)

In [62]:
COMUNE = "San Gervasio Bresciano"

totale_maschi = out.loc[out[COL_COMUNE]==COMUNE,"Maschi"].sum()
totale_femmine = out.loc[out[COL_COMUNE]==COMUNE,"Femmine"].sum()

print(f"Totale Maschi: {totale_maschi}")
print(f"Totale Femmine: {totale_femmine}")
print(f"Totale popolazione: {totale_maschi + totale_femmine}")

Totale Maschi: 1337
Totale Femmine: 1342
Totale popolazione: 2679


In [63]:
OUTPUT_NAME = "residenti_lombardia_per_fascia.csv"
out.to_csv(OUTPUT_NAME, sep=SEP, index=False, encoding="utf-8-sig")